# COMP9085 Week 11: fine-tuning ESM-2 with LoRA (optional demonstration)

**What this is.** A small, complete fine-tuning of a protein language model, so that you can see every step: the
pretrained ESM-2 8M model (6 layers, 7.5 million parameters), LoRA adapters of rank 8 on the query and value
projections, a classification head, and the signal-peptide task of the lab, on the first 100 residues of each protein.
It reproduces the protocol of the recorded runs (`data/week11_finetune_results.json`) with **one seed, one learning rate
and four epochs** to keep it to a few minutes. **You are not expected to implement this independently.**

**Requirements.** `pip install torch transformers peft scikit-learn pandas` (CPU is enough; no GPU). The first run
downloads the model from Hugging Face (`facebook/esm2_t6_8M_UR50D`, about 30 MB, MIT licence). Runtime about 3–6
minutes on a laptop CPU.

In [1]:
import json, time
from pathlib import Path
import numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model

SEED, WINDOW, EPOCHS, LR, BATCH = 42, 100, 4, 2e-3, 16
MODEL, REVISION = "facebook/esm2_t6_8M_UR50D", "c731040fcd8d73dceaa04b0a8e6329b345b0f5df"
torch.manual_seed(0); torch.set_num_threads(8)
DATA = next(q for q in [Path("data"), Path("../data")] if (q / "swissprot_human_2000.csv").exists())
prot = pd.read_csv(DATA / "swissprot_human_2000.csv")
y, tm = prot.has_signal_peptide.values, prot.is_transmembrane.values
idx = np.arange(len(prot))
tr, te = train_test_split(idx, test_size=0.25, stratify=2 * tm + y, random_state=SEED)
fit, val = train_test_split(tr, test_size=0.2, stratify=y[tr], random_state=SEED)
win = [s[:WINDOW] for s in prot.sequence]
print(len(fit), "fit /", len(val), "validation /", len(te), "test proteins")

1200 fit / 300 validation / 500 test proteins


## 1. The model and the adapters

`get_peft_model` freezes every pretrained weight and adds, beside each query and value projection W (320 × 320), two
small matrices B (320 × 8) and A (8 × 320); the layer computes W·x + B·A·x. Only A, B and the new classification head are
trained.

In [2]:
tok = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, revision=REVISION, num_labels=2)
model = get_peft_model(model, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.1, target_modules=["query", "value"],
                                         modules_to_save=["classifier"], bias="none"))
model.print_trainable_parameters()

trainable params: 164,802 || all params: 7,677,245 || trainable%: 2.1466


## 2. Train on the fit proteins, watch the validation proteins

In [3]:
def batches(ix, shuffle=False, rng=None):
    ix = np.array(ix)
    if shuffle:
        ix = ix[rng.permutation(len(ix))]
    for i in range(0, len(ix), BATCH):
        b = ix[i:i + BATCH]
        yield tok([win[j] for j in b], return_tensors="pt", padding=True), torch.tensor(y[b])

def predict(ix):
    model.eval(); out = []
    with torch.no_grad():
        for enc, _ in batches(ix):
            out.append(torch.softmax(model(**enc).logits, -1)[:, 1].numpy())
    return np.concatenate(out)

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01)
rng = np.random.default_rng(0)
for ep in range(1, EPOCHS + 1):
    model.train(); t = time.perf_counter(); tot = 0
    for enc, yb in batches(fit, True, rng):
        loss = torch.nn.functional.cross_entropy(model(**enc).logits, yb)
        opt.zero_grad(); loss.backward(); opt.step(); tot += float(loss) * len(yb)
    print(f"epoch {ep}: training loss {tot / len(fit):.4f}  validation ROC AUC {roc_auc_score(y[val], predict(val)):.4f}"
          f"  ({time.perf_counter() - t:.0f} s)")

epoch 1: training loss 0.1667  validation ROC AUC 0.9969  (13 s)


epoch 2: training loss 0.0664  validation ROC AUC 0.9964  (13 s)


epoch 3: training loss 0.0701  validation ROC AUC 0.9965  (12 s)


epoch 4: training loss 0.0621  validation ROC AUC 0.9970  (12 s)


## 3. Compare with the recorded run

The recorded run trained up to 12 epochs at each of two learning rates, kept the best validation epoch, and repeated
the chosen setting with three seeds. Here we only compare validation curves; the test set belongs to the recorded,
pre-planned evaluation and is not touched by this demonstration.

In [4]:
rec = json.load(open(DATA / "week11_finetune_results.json"))["finetune"]["runs"]["lora"]
curve = next(t for t in rec["tuning"] if t["lr"] == LR)["curve"]
for e in curve[:EPOCHS]:
    print(f"recorded epoch {e['epoch']}: validation ROC AUC {e['val_auc']:.4f}")

recorded epoch 1: validation ROC AUC 0.9969
recorded epoch 2: validation ROC AUC 0.9964
recorded epoch 3: validation ROC AUC 0.9965
recorded epoch 4: validation ROC AUC 0.9970
